Preprocessing

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import time
import pandas as pd
from textblob import TextBlob
from transformers import pipeline
from pathlib import Path


In [ ]:
from functools import lru_cache

_lemmatizer = WordNetLemmatizer()


@lru_cache(maxsize=8)
def _stopword_set(language: str) -> frozenset:
    return frozenset(stopwords.words(language))


def clean_text(text: str, language: str = "english") -> str:
    """Lowercase, tokenize, drop stopwords and non-alpha tokens, lemmatize; return a single string.

    NLTK data required (run once if missing):
        import nltk
        nltk.download("punkt")
        nltk.download("stopwords")
        nltk.download("wordnet")
        nltk.download("omw-1.4")
    """
    if text is None:
        return ""
    s = str(text).strip().lower()
    if not s:
        return ""

    stops = _stopword_set(language)
    tokens = word_tokenize(s)
    out: list[str] = []
    for tok in tokens:
        if not tok.isalpha():
            continue
        if tok in stops:
            continue
        out.append(_lemmatizer.lemmatize(tok))
    return " ".join(out)

In [ ]:
def load_excel(
    path: str | Path,
    sheet_name: str | int | list[int | str] | None = 0,
    header: int | None = 0,
    engine: str = "openpyxl",
) -> pd.DataFrame | dict[int | str, pd.DataFrame]:
    """Load one or more sheets from an `.xlsx` file.

    Parameters
    ----------
    path : str or Path
        Path to the workbook.
    sheet_name : str, int, list, or None
        Sheet to read: name, 0-based index, list of names/indices, or None for all sheets.
    header : int or None
        Row to use as column names (None = no header, e.g. single-column eval sheets).
    engine : str
        `openpyxl` for `.xlsx` (default).

    Returns
    -------
    DataFrame, or dict of DataFrames if multiple sheets were requested.
    """
    p = Path(path).expanduser().resolve()
    if not p.is_file():
        raise FileNotFoundError(p)
    return pd.read_excel(p, sheet_name=sheet_name, header=header, engine=engine)

Ensemble Method
https://huggingface.co/mrm8488/t5-base-finetuned-sarcasm-twitter

In [ ]:
HF_SENTIMENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
HF_SENTIMENT_MODEL

def _scores_to_hf_outputs(scores: list[dict]) -> tuple[float, float, float, float, str]:
    """Pipeline scores (top_k=None) → P(neg), P(neu), P(pos), polarity, argmax label."""
    m = {str(s["label"]).lower(): float(s["score"]) for s in scores}
    neg = m.get("negative", m.get("label_0", 0.0))
    neu = m.get("neutral", m.get("label_1", 0.0))
    pos = m.get("positive", m.get("label_2", 0.0))
    polarity = pos - neg
    label = max(
        (("negative", neg), ("neutral", neu), ("positive", pos)), key=lambda x: x[1]
    )[0]
    return neg, neu, pos, polarity, label


def textblob_subjectivity(text: str) -> float:
    """Objective (0) → subjective (1); empty input → 0."""
    if text is None or not str(text).strip():
        return 0.0
    return float(TextBlob(str(text)).sentiment.subjectivity)


def hf_sentiment_batch(
    texts: list[str],
    *,
    model_name: str = HF_SENTIMENT_MODEL,
    batch_size: int = 16,
) -> tuple[list[float], list[float], list[float], list[float], list[str]]:
    """HF class probabilities, polarity (pos−neg), and argmax label."""
    if not texts:
        return [], [], [], [], []
    _empty_mask = [not str(t).strip() for t in texts]
    _feed = [t if str(t).strip() else " " for t in texts]
    clf = pipeline(
        "sentiment-analysis",
        model=model_name,
        tokenizer=model_name,
        truncation=True,
        max_length=512,
        top_k=None,
    )
    rows = clf(_feed, batch_size=batch_size)
    probs_neg: list[float] = []
    probs_neu: list[float] = []
    probs_pos: list[float] = []
    polarities: list[float] = []
    labels: list[str] = []
    for i, row in enumerate(rows):
        if _empty_mask[i]:
            probs_neg.append(0.0)
            probs_neu.append(1.0)
            probs_pos.append(0.0)
            polarities.append(0.0)
            labels.append("neutral")
            continue
        neg, neu, pos, pol, lab = _scores_to_hf_outputs(row)
        probs_neg.append(neg)
        probs_neu.append(neu)
        probs_pos.append(pos)
        polarities.append(pol)
        labels.append(lab)
    return probs_neg, probs_neu, probs_pos, polarities, labels